In [2]:
# Cell 1 — Imports
import pandas as pd
import psycopg

In [3]:
# Cell 2 — Load evaluated predictions

predictions = pd.read_csv("../data/nq_first_to_100_predictions_full.csv")

predictions["Date"] = pd.to_datetime(predictions["Date"])

print("Prediction rows:", len(predictions))
print("Date range:", predictions["Date"].min(), "to", predictions["Date"].max())
print("\nBias counts:")
print(predictions["Bias"].value_counts(dropna=False))

predictions.head()

Prediction rows: 512
Date range: 2024-09-02 00:00:00 to 2026-08-28 00:00:00

Bias counts:
Bias
Long     269
Short    243
Name: count, dtype: int64


,Date,Day,Bias,Confidence,Auction Direction,Context,Result,Correct,Notes
0,2024-09-02,Monday,Long,61%,Moderate Up,Trend Continuation,Invalid,NaN,Positive ORG followed an overnight sell-side s...
1,2024-09-03,Tuesday,Short,63%,Strong Down,Trend Continuation,Short,True,Large negative ORG accompanied a persistent de...
2,2024-09-04,Wednesday,Short,66%,Strong Down,Trend Continuation,Long,False,Large negative ORG followed an exceptionally w...
3,2024-09-05,Thursday,Short,59%,Moderate Down,Balance,Long,False,Negative ORG developed within a choppy overnig...
4,2024-09-06,Friday,Long,62%,Moderate Up,Exhaustion,Short,False,Nearly flat ORG followed a broad overnight ran...


In [4]:
# Cell 3 — Pull RTH candles and align to prediction dates

conn = psycopg.connect("dbname=dailyedge_development")

query = """
SELECT timestamp, open, high, low, close, volume
FROM CANDLES
WHERE timestamp >= %s
  AND timestamp <= %s
  AND timestamp::time BETWEEN '08:30:00' AND '15:15:00'
ORDER BY timestamp
"""

candles = pd.read_sql(
    query,
    conn,
    params=(predictions["Date"].min(), predictions["Date"].max())
)
conn.close()

candles["timestamp"] = pd.to_datetime(candles["timestamp"])
candles["Date"] = candles["timestamp"].dt.normalize()

print("RTH candles:", len(candles))
print("Sessions:", candles["Date"].nunique())
print("Date range:", candles["Date"].min(), "to", candles["Date"].max())

/tmp/ipykernel_46093/1575524380.py:14: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  candles = pd.read_sql(


RTH candles: 204218
Sessions: 512
Date range: 2024-09-02 00:00:00 to 2026-08-27 00:00:00


In [5]:
# Cell 4 — Match prediction dates to available, complete candle sessions

prediction_dates = set(predictions["Date"].dropna())
candle_dates = set(candles["Date"].dropna())

matched_dates = prediction_dates & candle_dates

matched_predictions = predictions[
    predictions["Date"].isin(matched_dates)
].copy()

matched_candles = candles[
    candles["Date"].isin(matched_dates)
].copy()

session_validation = (
    matched_candles.groupby("Date")
    .agg(
        first_time=("timestamp", "min"),
        last_time=("timestamp", "max"),
        candle_count=("timestamp", "size")
    )
)

session_validation["has_open"] = (
    session_validation["first_time"].dt.time ==
    pd.Timestamp("08:30:00").time()
)

session_validation["has_close"] = (
    session_validation["last_time"].dt.time ==
    pd.Timestamp("15:15:00").time()
)

print("Prediction dates:", len(prediction_dates))
print("Candle dates:", len(candle_dates))
print("Matched dates:", len(matched_dates))

print("Matched prediction/session dates:", len(matched_predictions))
print("Sessions with 08:30 open:", session_validation["has_open"].sum())
print("Sessions with 15:15 close:", session_validation["has_close"].sum())

print(
    "Fully valid sessions:",
    (session_validation["has_open"] & session_validation["has_close"]).sum()
)

session_validation[
    ~(session_validation["has_open"] & session_validation["has_close"])
]

Prediction dates: 512
Candle dates: 512
Matched dates: 509
Matched prediction/session dates: 509
Sessions with 08:30 open: 509
Sessions with 15:15 close: 491
Fully valid sessions: 491


,first_time,last_time,candle_count,has_open,has_close
Date,,,,,
2024-09-02,2024-09-02 08:30:00,2024-09-02 11:59:00,210,True,False
2024-11-28,2024-11-28 08:30:00,2024-11-28 11:59:00,210,True,False
2024-11-29,2024-11-29 08:30:00,2024-11-29 12:14:00,225,True,False
2024-12-24,2024-12-24 08:30:00,2024-12-24 12:14:00,225,True,False
2025-02-17,2025-02-17 08:30:00,2025-02-17 11:59:00,210,True,False
2025-05-26,2025-05-26 08:30:00,2025-05-26 11:59:00,210,True,False
2025-06-19,2025-06-19 08:30:00,2025-06-19 11:59:00,210,True,False
2025-07-03,2025-07-03 08:30:00,2025-07-03 12:14:00,225,True,False
2025-07-04,2025-07-04 08:30:00,2025-07-04 11:59:00,210,True,False


In [6]:
# Cell 5 — Restrict to fully valid RTH sessions

valid_dates = session_validation[
    session_validation["has_open"] &
    session_validation["has_close"]
].index

study_predictions = matched_predictions[
    matched_predictions["Date"].isin(valid_dates)
].copy()

study_candles = matched_candles[
    matched_candles["Date"].isin(valid_dates)
].copy()

print("Study prediction rows:", len(study_predictions))
print("Study candle sessions:", study_candles["Date"].nunique())
print("Study RTH candles:", len(study_candles))

print("\nBias counts in study set:")
print(study_predictions["Bias"].value_counts(dropna=False))

Study prediction rows: 491
Study candle sessions: 491
Study RTH candles: 199341

Bias counts in study set:
Bias
Long     256
Short    235
Name: count, dtype: int64


In [7]:
# Cell 6 — Pyramid / conditional-trail trade simulator

def simulate_pyramid_trade(
    day_df,
    direction,
    initial_sl_pts=100,
    initial_tp_pts=100,
    adverse_add_pts=50,
    favorable_add_pts=50,
    trail_shift_pts=50,
):
    """
    Rule 1: enter 1 unit at session open in `direction`. SL = initial_sl_pts,
            TP = initial_tp_pts.
    Rule 2: if price moves `adverse_add_pts` against the position first,
            add 1 unit at that price. SL and TP are unchanged.
    Rule 3: if price moves `favorable_add_pts` in favor first, add 1 unit at
            that price, and shift the SL by `trail_shift_pts` in the
            favorable direction (e.g. long: entry-100 -> entry-50 -- this
            shifts the stop by the trail amount, it does NOT jump it to
            breakeven). TP is unchanged.
    At most one add per session (rule 2 and rule 3 are mutually exclusive).

    1-min OHLC can't establish intrabar order when:
      - SL and TP are both touched in the same bar, or
      - the adverse and favorable triggers are both touched in the same bar
        before any add has happened
    Both cases return status "Unknown" (pnl=None).

    When a single bar's low/high crosses BOTH an add trigger and the
    corresponding SL/TP beyond it (e.g. a down bar's low crosses -50 and
    -100), price must have passed the nearer trigger first, so we
    deterministically add THEN exit at SL/TP.

    An open-price gap directly through SL or TP is resolved as an immediate
    exit at that level with no add (a gap has no tradeable price at which
    the add could have filled).

    Returns a dict: status, pnl (total points summed across all units,
    None if Unknown), added (bool), add_type ("adverse"/"favorable"/None),
    n_units.
    """
    sign = 1 if direction == "Long" else -1
    open_price = day_df.iloc[0]["open"]

    sl = open_price - sign * initial_sl_pts
    tp = open_price + sign * initial_tp_pts
    adverse_trigger = open_price - sign * adverse_add_pts
    favorable_trigger = open_price + sign * favorable_add_pts

    unit_entries = [open_price]
    added = False
    add_type = None

    def pnl_at(price):
        return sum(sign * (price - e) for e in unit_entries)

    for _, candle in day_df.iterrows():
        o, h, l = candle["open"], candle["high"], candle["low"]

        sl_hit = (l <= sl) if sign == 1 else (h >= sl)
        tp_hit = (h >= tp) if sign == 1 else (l <= tp)

        gapped_sl = (o <= sl) if sign == 1 else (o >= sl)
        gapped_tp = (o >= tp) if sign == 1 else (o <= tp)

        if gapped_sl:
            return {"status": "Stopped", "pnl": pnl_at(sl), "added": added,
                    "add_type": add_type, "n_units": len(unit_entries)}
        if gapped_tp:
            return {"status": "Target", "pnl": pnl_at(tp), "added": added,
                    "add_type": add_type, "n_units": len(unit_entries)}

        if not added:
            adverse_hit = (l <= adverse_trigger) if sign == 1 else (h >= adverse_trigger)
            favorable_hit = (h >= favorable_trigger) if sign == 1 else (l <= favorable_trigger)

            if sl_hit and tp_hit:
                return {"status": "Unknown", "pnl": None, "added": added,
                        "add_type": add_type, "n_units": len(unit_entries)}

            if adverse_hit and favorable_hit and not sl_hit and not tp_hit:
                return {"status": "Unknown", "pnl": None, "added": added,
                        "add_type": add_type, "n_units": len(unit_entries)}

            if tp_hit:
                unit_entries.append(favorable_trigger)
                sl = sl + sign * trail_shift_pts
                return {"status": "Target", "pnl": pnl_at(tp), "added": True,
                        "add_type": "favorable", "n_units": len(unit_entries)}

            if sl_hit:
                unit_entries.append(adverse_trigger)
                return {"status": "Stopped", "pnl": pnl_at(sl), "added": True,
                        "add_type": "adverse", "n_units": len(unit_entries)}

            if favorable_hit:
                unit_entries.append(favorable_trigger)
                sl = sl + sign * trail_shift_pts
                added, add_type = True, "favorable"
                continue

            if adverse_hit:
                unit_entries.append(adverse_trigger)
                added, add_type = True, "adverse"
                continue

        else:
            if sl_hit and tp_hit:
                return {"status": "Unknown", "pnl": None, "added": added,
                        "add_type": add_type, "n_units": len(unit_entries)}
            if tp_hit:
                return {"status": "Target", "pnl": pnl_at(tp), "added": added,
                        "add_type": add_type, "n_units": len(unit_entries)}
            if sl_hit:
                return {"status": "Stopped", "pnl": pnl_at(sl), "added": added,
                        "add_type": add_type, "n_units": len(unit_entries)}

    final_close = day_df.iloc[-1]["close"]
    return {"status": "Close", "pnl": pnl_at(final_close), "added": added,
            "add_type": add_type, "n_units": len(unit_entries)}

In [8]:
# Cell 7 — Run the pyramid simulation over all study predictions

pyramid_results = []

for _, prediction in study_predictions.iterrows():
    date = prediction["Date"]
    direction = prediction["Bias"]

    day_df = study_candles[
        study_candles["Date"] == date
    ]

    if direction in ("Long", "Short"):
        outcome = simulate_pyramid_trade(day_df, direction)
    else:
        outcome = {"status": "Unknown", "pnl": None, "added": False,
                   "add_type": None, "n_units": 1}

    pyramid_results.append({
        "Date": date,
        "Day": prediction["Day"],
        "Bias": direction,
        "Status": outcome["status"],
        "PnL": outcome["pnl"],
        "Added": outcome["added"],
        "Add_Type": outcome["add_type"],
        "N_Units": outcome["n_units"],
    })

pyramid_trades = pd.DataFrame(pyramid_results)

print("Total simulated trades:", len(pyramid_trades))
print("\nStatus counts:")
print(pyramid_trades["Status"].value_counts(dropna=False))
print("\nAdd type counts:")
print(pyramid_trades["Add_Type"].value_counts(dropna=False))
print("\nUnknown rate:", (pyramid_trades["Status"] == "Unknown").mean().round(4))

Total simulated trades: 491

Status counts:
Status
Stopped    248
Target     228
Close       13
Unknown      2
Name: count, dtype: int64

Add type counts:
Add_Type
favorable    269
adverse      220
NaN            2
Name: count, dtype: int64

Unknown rate: 0.0041


In [9]:
# Cell 8 — Expectancy by day of week

weekday_order = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday"]

resolved_trades = pyramid_trades.dropna(subset=["PnL"])

expectancy_by_day = (
    resolved_trades
    .groupby("Day")["PnL"]
    .agg(
        Trades="size",
        Win_Rate=lambda s: (s > 0).mean() * 100,
        Avg_Win=lambda s: s[s > 0].mean(),
        Avg_Loss=lambda s: s[s < 0].mean(),
        Expectancy="mean",
        Total_PnL="sum",
    )
    .reindex(weekday_order)
)

expectancy_by_day.round(2)

,Trades,Win_Rate,Avg_Win,Avg_Loss,Expectancy,Total_PnL
Day,,,,,,
Monday,96,48.96,162.49,-142.03,7.06,677.49
Tuesday,101,48.51,171.00,-143.28,9.19,928.59
Wednesday,101,52.48,168.87,-150.00,17.33,1750.00
Thursday,97,35.05,178.41,-149.56,-34.60,-3356.47
Friday,94,52.13,169.59,-144.68,19.14,1799.17


In [13]:
# Cell 9 — Retracement-only entry simulator (no initial entry, no add)
# CORRECTED: SL/TP stay anchored to the original session open, not the entry price.
# So actual risk from entry = sl_pts - retracement_pts = 50pts,
# and actual reward from entry = tp_pts + retracement_pts = 150pts.

def simulate_retracement_entry_trade(
    day_df,
    direction,
    retracement_pts=50,
    sl_pts=100,
    tp_pts=100,
):
    """
    No trade at the session open. Wait for price to move `retracement_pts`
    against the predicted `direction` (measured from the session open) and
    enter 1 unit exactly at that price. SL and TP remain anchored to the
    ORIGINAL SESSION OPEN at sl_pts/tp_pts (not to the entry price) -- so
    for the default 100/100 with a 50pt retracement entry, actual risk is
    50pts and actual reward is 150pts. No pyramiding.

    If the retracement never happens before session end, the day has no
    trade ("No_Entry").

    Ambiguity handling mirrors the pyramid simulator: SL and TP touched in
    the same bar -> "Unknown"; a bar whose low/high crosses both the entry
    trigger and SL/TP beyond it is resolved deterministically (entry then
    exit, since price must pass the nearer level first); an opening gap
    that jumps through a level is resolved at that level's price.
    """
    sign = 1 if direction == "Long" else -1
    open_price = day_df.iloc[0]["open"]
    entry_trigger = open_price - sign * retracement_pts
    sl = open_price - sign * sl_pts
    tp = open_price + sign * tp_pts

    entered = False
    entry_price = None

    for _, candle in day_df.iterrows():
        o, h, l = candle["open"], candle["high"], candle["low"]

        if not entered:
            gapped_entry = (o <= entry_trigger) if sign == 1 else (o >= entry_trigger)
            entry_hit = (l <= entry_trigger) if sign == 1 else (h >= entry_trigger)

            if not (gapped_entry or entry_hit):
                continue

            entry_price = entry_trigger
            entered = True

            sl_hit = (l <= sl) if sign == 1 else (h >= sl)
            tp_hit = (h >= tp) if sign == 1 else (l <= tp)
            gapped_sl = (o <= sl) if sign == 1 else (o >= sl)
            gapped_tp = (o >= tp) if sign == 1 else (o <= tp)

            if (gapped_sl or sl_hit) and (gapped_tp or tp_hit):
                return {"status": "Unknown", "pnl": None, "entered": True}
            if gapped_sl or sl_hit:
                return {"status": "Stopped", "pnl": sign * (sl - entry_price), "entered": True}
            if gapped_tp or tp_hit:
                return {"status": "Target", "pnl": sign * (tp - entry_price), "entered": True}

            continue

        else:
            sl_hit = (l <= sl) if sign == 1 else (h >= sl)
            tp_hit = (h >= tp) if sign == 1 else (l <= tp)
            gapped_sl = (o <= sl) if sign == 1 else (o >= sl)
            gapped_tp = (o >= tp) if sign == 1 else (o <= tp)

            if gapped_sl:
                return {"status": "Stopped", "pnl": sign * (sl - entry_price), "entered": True}
            if gapped_tp:
                return {"status": "Target", "pnl": sign * (tp - entry_price), "entered": True}
            if sl_hit and tp_hit:
                return {"status": "Unknown", "pnl": None, "entered": True}
            if tp_hit:
                return {"status": "Target", "pnl": sign * (tp - entry_price), "entered": True}
            if sl_hit:
                return {"status": "Stopped", "pnl": sign * (sl - entry_price), "entered": True}

    if not entered:
        return {"status": "No_Entry", "pnl": None, "entered": False}

    final_close = day_df.iloc[-1]["close"]
    return {"status": "Close", "pnl": sign * (final_close - entry_price), "entered": True}

In [14]:
# Cell 10 — Run the retracement-only simulation over all study predictions

retracement_results = []

for _, prediction in study_predictions.iterrows():
    date = prediction["Date"]
    direction = prediction["Bias"]

    day_df = study_candles[
        study_candles["Date"] == date
    ]

    if direction in ("Long", "Short"):
        outcome = simulate_retracement_entry_trade(day_df, direction)
    else:
        outcome = {"status": "Unknown", "pnl": None, "entered": False}

    retracement_results.append({
        "Date": date,
        "Day": prediction["Day"],
        "Bias": direction,
        "Status": outcome["status"],
        "PnL": outcome["pnl"],
        "Entered": outcome["entered"],
    })

retracement_trades = pd.DataFrame(retracement_results)

print("Total prediction days:", len(retracement_trades))
print("\nStatus counts:")
print(retracement_trades["Status"].value_counts(dropna=False))
print("\nEntry rate:", retracement_trades["Entered"].mean().round(4))
print("Unknown rate (among entered):", (
    retracement_trades.loc[retracement_trades["Entered"], "Status"] == "Unknown"
).mean().round(4))

Total prediction days: 491

Status counts:
Status
Stopped     255
No_Entry    119
Target       90
Close        27
Name: count, dtype: int64

Entry rate: 0.7576
Unknown rate (among entered): 0.0


In [15]:
# Cell 11 — Expectancy by day of week (retracement-only study)

resolved_retracement_trades = retracement_trades.dropna(subset=["PnL"])

retracement_expectancy_by_day = (
    resolved_retracement_trades
    .groupby("Day")["PnL"]
    .agg(
        Trades="size",
        Win_Rate=lambda s: (s > 0).mean() * 100,
        Avg_Win=lambda s: s[s > 0].mean(),
        Avg_Loss=lambda s: s[s < 0].mean(),
        Expectancy="mean",
        Total_PnL="sum",
    )
    .reindex(weekday_order)
)

retracement_expectancy_by_day.round(2)

,Trades,Win_Rate,Avg_Win,Avg_Loss,Expectancy,Total_PnL
Day,,,,,,
Monday,68,29.41,115.66,-50.00,-1.28,-86.88
Tuesday,78,33.33,131.02,-48.63,11.26,877.96
Wednesday,69,30.43,140.52,-50.00,7.98,550.88
Thursday,86,24.42,140.46,-49.79,-3.33,-286.61
Friday,71,35.21,125.67,-49.85,11.95,848.76


In [17]:
# Cell 12 — Retracement entry + return-to-open add simulator

def simulate_retracement_reentry_trade(
    day_df,
    direction,
    retracement_pts=50,
    sl_pts=100,
    tp_pts=100,
):
    """
    Unit 1 enters only after price first pulls back `retracement_pts`
    against the predicted `direction` (measured from the session open) --
    same entry rule as the retracement-only study. SL and TP are fixed at
    sl_pts/tp_pts from the ORIGINAL SESSION OPEN (not the entry price), so
    unit 1's actual initial risk is (sl_pts - retracement_pts) and actual
    initial reward is (tp_pts + retracement_pts).

    NEW: if price later comes back to the original session open (before
    hitting SL or TP), add a second unit exactly at the open price. SL and
    TP remain unchanged -- no trailing -- so after the add, unit 2's risk
    is sl_pts and reward is tp_pts.

    If the initial retracement never happens, the day has no trade
    ("No_Entry"). Ambiguity handling follows the same conventions as the
    other simulators in this notebook (simultaneous SL/opposite-side level
    in one bar -> "Unknown"; a bar crossing both a nearer and farther level
    on the same side is resolved deterministically; a bar's open already
    past a level is treated as reaching that level).
    """
    sign = 1 if direction == "Long" else -1
    open_price = day_df.iloc[0]["open"]
    entry_trigger = open_price - sign * retracement_pts
    sl = open_price - sign * sl_pts
    tp = open_price + sign * tp_pts
    add_price = open_price

    entered = False
    added = False
    unit_entries = []

    def pnl_at(price):
        return sum(sign * (price - e) for e in unit_entries)

    for _, candle in day_df.iterrows():
        o, h, l = candle["open"], candle["high"], candle["low"]

        if not entered:
            gapped_entry = (o <= entry_trigger) if sign == 1 else (o >= entry_trigger)
            entry_hit = (l <= entry_trigger) if sign == 1 else (h >= entry_trigger)

            if not (gapped_entry or entry_hit):
                continue

            entered = True
            unit_entries.append(entry_trigger)
            # fall through: resolve the rest of this same bar below

        sl_hit = (l <= sl) if sign == 1 else (h >= sl)
        tp_hit = (h >= tp) if sign == 1 else (l <= tp)
        gapped_sl = (o <= sl) if sign == 1 else (o >= sl)
        gapped_tp = (o >= tp) if sign == 1 else (o <= tp)

        if not added:
            add_hit = (h >= add_price) if sign == 1 else (l <= add_price)
            gapped_add = (o >= add_price) if sign == 1 else (o <= add_price)

            if (gapped_sl or sl_hit) and (gapped_add or add_hit or gapped_tp or tp_hit):
                return {"status": "Unknown", "pnl": None, "entered": True, "added": added}

            if gapped_tp or tp_hit:
                unit_entries.append(add_price)
                return {"status": "Target", "pnl": pnl_at(tp), "entered": True, "added": True}

            if gapped_sl or sl_hit:
                return {"status": "Stopped", "pnl": pnl_at(sl), "entered": True, "added": False}

            if gapped_add or add_hit:
                unit_entries.append(add_price)
                added = True
                continue

            continue

        else:
            if (gapped_sl or sl_hit) and (gapped_tp or tp_hit):
                return {"status": "Unknown", "pnl": None, "entered": True, "added": True}
            if gapped_tp or tp_hit:
                return {"status": "Target", "pnl": pnl_at(tp), "entered": True, "added": True}
            if gapped_sl or sl_hit:
                return {"status": "Stopped", "pnl": pnl_at(sl), "entered": True, "added": True}

    if not entered:
        return {"status": "No_Entry", "pnl": None, "entered": False, "added": False}

    final_close = day_df.iloc[-1]["close"]
    return {"status": "Close", "pnl": pnl_at(final_close), "entered": True, "added": added}

In [18]:
# Cell 13 — Run the retracement + return-to-open add simulation

reentry_results = []

for _, prediction in study_predictions.iterrows():
    date = prediction["Date"]
    direction = prediction["Bias"]

    day_df = study_candles[
        study_candles["Date"] == date
    ]

    if direction in ("Long", "Short"):
        outcome = simulate_retracement_reentry_trade(day_df, direction)
    else:
        outcome = {"status": "Unknown", "pnl": None, "entered": False, "added": False}

    reentry_results.append({
        "Date": date,
        "Day": prediction["Day"],
        "Bias": direction,
        "Status": outcome["status"],
        "PnL": outcome["pnl"],
        "Entered": outcome["entered"],
        "Added": outcome["added"],
    })

reentry_trades = pd.DataFrame(reentry_results)

print("Total prediction days:", len(reentry_trades))
print("\nStatus counts:")
print(reentry_trades["Status"].value_counts(dropna=False))
print("\nEntry rate:", reentry_trades["Entered"].mean().round(4))
print("Add rate (among entered):", (
    reentry_trades.loc[reentry_trades["Entered"], "Added"]
).mean().round(4))
print("Unknown rate (among entered):", (
    reentry_trades.loc[reentry_trades["Entered"], "Status"] == "Unknown"
).mean().round(4))

Total prediction days: 491

Status counts:
Status
Stopped     251
No_Entry    119
Target       90
Close        27
Unknown       4
Name: count, dtype: int64

Entry rate: 0.7576
Add rate (among entered): 0.5403
Unknown rate (among entered): 0.0108


In [19]:
# Cell 14 — Expectancy by day of week (retracement + return-to-open add study)

resolved_reentry_trades = reentry_trades.dropna(subset=["PnL"])

reentry_expectancy_by_day = (
    resolved_reentry_trades
    .groupby("Day")["PnL"]
    .agg(
        Trades="size",
        Win_Rate=lambda s: (s > 0).mean() * 100,
        Avg_Win=lambda s: s[s > 0].mean(),
        Avg_Loss=lambda s: s[s < 0].mean(),
        Expectancy="mean",
        Total_PnL="sum",
    )
    .reindex(weekday_order)
)

reentry_expectancy_by_day.round(2)

,Trades,Win_Rate,Avg_Win,Avg_Loss,Expectancy,Total_PnL
Day,,,,,,
Monday,68,26.47,202.42,-80.35,-5.50,-373.75
Tuesday,78,30.77,233.53,-79.13,17.07,1331.77
Wednesday,67,31.34,231.04,-89.13,11.22,751.77
Thursday,86,24.42,230.92,-83.42,-6.67,-573.21
Friday,69,34.78,212.99,-89.03,16.02,1105.60


In [20]:
# Cell 15 — Effective_Bias: contrarian on Thursdays

def flip_bias(row):
    if row["Day"] == "Thursday" and row["Bias"] in ("Long", "Short"):
        return "Short" if row["Bias"] == "Long" else "Long"
    return row["Bias"]

study_predictions["Effective_Bias"] = study_predictions.apply(flip_bias, axis=1)

print("Bias vs Effective_Bias mismatches (should all be Thursday):")
print(
    study_predictions.loc[
        study_predictions["Bias"] != study_predictions["Effective_Bias"],
        "Day"
    ].value_counts()
)

Bias vs Effective_Bias mismatches (should all be Thursday):
Day
Thursday    97
Name: count, dtype: int64


In [21]:
# Cell 16 — Run the retracement + return-to-open add simulation with contrarian Thursday

contrarian_reentry_results = []

for _, prediction in study_predictions.iterrows():
    date = prediction["Date"]
    direction = prediction["Effective_Bias"]

    day_df = study_candles[
        study_candles["Date"] == date
    ]

    if direction in ("Long", "Short"):
        outcome = simulate_retracement_reentry_trade(day_df, direction)
    else:
        outcome = {"status": "Unknown", "pnl": None, "entered": False, "added": False}

    contrarian_reentry_results.append({
        "Date": date,
        "Day": prediction["Day"],
        "Bias": prediction["Bias"],
        "Effective_Bias": direction,
        "Status": outcome["status"],
        "PnL": outcome["pnl"],
        "Entered": outcome["entered"],
        "Added": outcome["added"],
    })

contrarian_reentry_trades = pd.DataFrame(contrarian_reentry_results)

print("Total prediction days:", len(contrarian_reentry_trades))
print("\nStatus counts:")
print(contrarian_reentry_trades["Status"].value_counts(dropna=False))
print("\nEntry rate:", contrarian_reentry_trades["Entered"].mean().round(4))
print("Add rate (among entered):", (
    contrarian_reentry_trades.loc[contrarian_reentry_trades["Entered"], "Added"]
).mean().round(4))


Total prediction days: 491

Status counts:
Status
Stopped     234
No_Entry    138
Target       89
Close        26
Unknown       4
Name: count, dtype: int64

Entry rate: 0.7189
Add rate (among entered): 0.5694


In [22]:
# Cell 17 — Expectancy by day of week (contrarian-Thursday retracement+add study)

resolved_contrarian_trades = contrarian_reentry_trades.dropna(subset=["PnL"])

contrarian_expectancy_by_day = (
    resolved_contrarian_trades
    .groupby("Day")["PnL"]
    .agg(
        Trades="size",
        Win_Rate=lambda s: (s > 0).mean() * 100,
        Avg_Win=lambda s: s[s > 0].mean(),
        Avg_Loss=lambda s: s[s < 0].mean(),
        Expectancy="mean",
        Total_PnL="sum",
    )
    .reindex(weekday_order)
)

contrarian_expectancy_by_day.round(2)

,Trades,Win_Rate,Avg_Win,Avg_Loss,Expectancy,Total_PnL
Day,,,,,,
Monday,68,26.47,202.42,-80.35,-5.50,-373.75
Tuesday,78,30.77,233.53,-79.13,17.07,1331.77
Wednesday,67,31.34,231.04,-89.13,11.22,751.77
Thursday,67,28.36,228.71,-101.29,-7.71,-516.24
Friday,69,34.78,212.99,-89.03,16.02,1105.60
